# First All-photonic Repeater Network

This example will demonstrate how to use QNPack to simulate a quantum network repeater chain using a single quantum repeater (QR).

<img src="./images/ape_network.png" width="75%"><br>



The APE repeater network is consisted of quantum end nodes (Q-nodes), quantum repeaters (QRs), Bell-State-Measurement nodes (BSM),  classical and quantum channels. In the beginning, the emitter qubit in APE QRs will emit Repeater Graph State (RGS) following a predefined circuit. Since each APE QR generates and transmits RGS photonic qubits sequentially in a predefined manner, each photonic qubit arriving at a BSM-node in the designated time interval can be identified as either a leaf, 1st-level core or 2nd-level core qubit. We make the following assumptions of photon loss and noise:
1. The sources of photon loss include collection efficiency, quantum frequency conversion efficiency, fiber attenuation, and detector efficiency.
2. The only noises are emitter (quantum dot) decoherence in APE QR and memory (nuclear spin) decoherence in Q-nodes.

In the code below, we first demonstrate how to use QNPack to simulate a quantum network repeater chain under an ideal situation without loss and noise.

In [ ]:
from qnpack.APE.allphotonic import APESimulation

fixed_params = {
    "network": {"photon_loss": 0},
    "emitter":{"emitter_T2":0, "memory_T2":0,"QFC_loss":0},    
    "rgs":{"num_branches_half":2,"b0":1,"b1":0} 
}
varying_params = {
    "num_repeaters": [1,2,3,4],
    "distance": [10]
}

sim = APESimulation(fixed_params=fixed_params,
                    varying_params=varying_params,
                    iterations=30,
                    min_successful=0,
                    parameter_file="parameters/scenario1.yml",
                    output_dir="results")
data=sim.start()
sim.plot(final_data=data, show_theo_rate=True, x_axis1="num_repeaters", y_axis1="sim_rate", xlabel1="Number of repeaters", ylabel1="Rate (Hz)",
     label_param1="distance", title1="Rate vs Number of repeaters",
     x_axis2="num_repeaters", y_axis2="fidelity", xlabel2="Number of repeaters", ylabel2="Fidelity",
     label_param2="distance", title2="Fidelity vs Number of repeaters"
     )
sim.finalize()



# Effect of number of branches in RGS:
We can see under the ideal situation, the repeater rate is actually decreasing with number of repeaters (QRs). This is because n repeaters would require n+1 BSM nodes, and a repeater chain would succeed only if at least one BSM succeeds on each BSM node. For each BSM, the success probability would involve two photons arriving from the left and right nodes, multiplied by the success probability 0.5 by linear optics BSM: 

$P_{BSM}=(1-P_{loss})^2/2$

where $P_{loss}$ is the loss probability of a single photon.
For a repeater graph state, there are a total number of m BSMs in each BSM node, which corresponds to the parameter `num_branches_half` in the code. The probability of at least one of them succeeds is:

$P_{BSM,\geq1}=1-(1-P_{BSM})^m$

Hence, the probability of having at least one BSM succeeds at each BSM node in the whole repeater chain will decrease with the number of repeaters n:

$P_{BSM,chain}=(P_{BSM,\geq1})^{n+1}$

To solve this problem, we would need to increase the number of branches in the RGS, as shown in the code below.

In [ ]:
#from qnpack.APE.allphotonic import APESimulation

fixed_params = {
    "network": {"photon_loss": 0},
    "emitter":{"emitter_T2":0, "memory_T2":0,"QFC_loss":0},
    "rgs":{"num_branches_half":6,"b0":1,"b1":0}    
}
varying_params = {
    "num_repeaters": [1,2,3,4],
    "distance": [10]
}

sim = APESimulation(fixed_params=fixed_params,
                    varying_params=varying_params,
                    iterations=30,
                    min_successful=0,
                    parameter_file="parameters/scenario1.yml",
                    output_dir="results")
data=sim.start()
sim.plot(final_data=data, show_theo_rate=True, x_axis1="num_repeaters", y_axis1="sim_rate", xlabel1="Number of repeaters", ylabel1="Rate (Hz)",
     label_param1="distance", title1="Rate vs Number of repeaters",
     x_axis2="num_repeaters", y_axis2="fidelity", xlabel2="Number of repeaters", ylabel2="Fidelity",
     label_param2="distance", title2="Fidelity vs Number of repeaters"
     )
sim.finalize()

# Effect of photon loss
After using RGS with more branches, since $P_{BSM,\geq1}$ would be increased, having more repeaters can actually increase the repeater rate now because the node-to-node distance is much shorter for the transmission of photon. 
Next, we introduce photon loss due to fiber attenuation and see how the repeater performance will be affected. The photon loss probability scales exponentially with length L of the quantum channel:
$P_{loss}=10^{-\alpha/L}$ where $\alpha$ is the signal attenuation rate in 0.2dB/km. The following code introduces photon loss under the same repeater setting from above.


In [ ]:
from qnpack.APE.allphotonic import APESimulation

fixed_params = {
    "network": {"photon_loss": 0.2},
    "emitter":{"emitter_T2":0, "memory_T2":0,"QFC_loss":0},
    "rgs":{"num_branches_half":6,"b0":1,"b1":0}    
}
varying_params = {
    "num_repeaters": [1,2,3,4],
    "distance": [10]
}

sim = APESimulation(fixed_params=fixed_params,
                    varying_params=varying_params,
                    iterations=30,
                    min_successful=0,
                    parameter_file="parameters/scenario1.yml",
                    output_dir="results")
data=sim.start()
sim.plot(final_data=data, show_theo_rate=True, x_axis1="num_repeaters", y_axis1="sim_rate", xlabel1="Number of repeaters", ylabel1="Rate (Hz)",
     label_param1="distance", title1="Rate vs Number of repeaters",
     x_axis2="num_repeaters", y_axis2="fidelity", xlabel2="Number of repeaters", ylabel2="Fidelity",
     label_param2="distance", title2="Fidelity vs Number of repeaters"
     )
sim.finalize()

# Tree-encoded RGS and fault tolerent measurement
We can see that the repeater rate decreases drastically when fiber loss is introduced, and having more repeaters makes the rate worse again! This is because the RGS scheme requires all the measurement of the core qubits to be successful. In the RGS, each core qubit plays the role of the trapped ion. All the core qubits are fully connected with one another in the graph. This is as if we perform the DBSM of trapped-ion repeater in advance, and perform Bell State measurement between photons emitted by trapped-ion afterwards. But in the APE case, the leaf qubits play the role of those photons emitted by trapped-ions. To understand the price we need to pay to replace the trapped-ion memory qubits by the core photons in the RGS, the success probability of receiving all the 2m core qubits in each BSM node is

$P_{all\: core}=(1-P_{loss})^{2m}$ 

and it also scales up with the number of repeaters n:

$P_{all\: core,chain}=(P_{all\: core})^{n+1}$

Hence, taking into account the probalistic BSM, the total success probablity for the whole repeater chain becomes

$P_{chain}=P_{BSM,chain}*P_{all\: core,chain}$

The requirement of all core qubits to be successfully measured is formiddable once m and n become large. This gives the motivation of introducing encoding to protect the core qubits. In the RGS, each core qubit is encoded by a tree, as shown in the following diagram:

<img src="./images/GS_tree_encoding.png" width="40%"><br>

The tree encoding uses a large number of photons to encode each core qubit, and the redunduncy provides protection against the loss of a certain subset of those photons. The decoding details of the tree encoding is implemented in the `NodeProtocol` of the control node in NetSquid, following the scheme in Physical review letters, 97(12), 120501. The code below shows how we can set up the 2-level tree-encoding of RGS, with a total of 12 branches, each consisted of a tree with parameters $b_0=6$ and $b_1=3$. Notice that this would require longer time for the simulation to run, becasue each (6,6,3) RGS is consisted of 300 photonic qubits, and for 4 repeaters, there would be over a thousand photonic qubits. This is handled by the `StabRepr` in NetSquid.

In [ ]:
from qnpack.APE.allphotonic import APESimulation

fixed_params = {
    "network": {"photon_loss": 0.2},
    "emitter":{"emitter_T2":0, "memory_T2":0,"QFC_loss":0},    
    "rgs":{"num_branches_half":6,"b0":6,"b1":3}
}
varying_params = {
    "num_repeaters": [1,2,3,4],
    "distance": [10]
}

sim = APESimulation(fixed_params=fixed_params,
                    varying_params=varying_params,
                    iterations=10,
                    min_successful=0,
                    parameter_file="parameters/scenario1.yml",
                    output_dir="results")
data=sim.start()
sim.plot(final_data=data, show_theo_rate=True, x_axis1="num_repeaters", y_axis1="sim_rate", xlabel1="Number of repeaters", ylabel1="Rate (Hz)",
     label_param1="distance", title1="Rate vs Number of repeaters",
     x_axis2="num_repeaters", y_axis2="fidelity", xlabel2="Number of repeaters", ylabel2="Fidelity",
     label_param2="distance", title2="Fidelity vs Number of repeaters"
     )
sim.finalize()

# Effect of quantum emitter coherence time on fidelity:
With the tree encoding, the repeater rate has increased almost to the level before we introduced photon loss. In general, a larger encoding would require a longer RGS generation time, but has a better ability to protect against photon loss, so a larger RGS does not necessarily give better performance. In addition, there are also huge resource overhead for generating such a large entangled photonic state. Inside NetSquid, we simulate the deterministic approach of graph state generation by sequentially stimulating the quantum emitter in each APE QR. The major source of noise comes from the decoherence effect of the quantum emitter during graph state generation. In the code below, we investigate how the docoherence effect impact the fidelity of the final entangled Bell state across the two ends, with the coherence time of the quantum dot emitter set to $3\mu s$, and the coherence time of the nuclear spin memory set to $20ms$.  

In [ ]:
from qnpack.APE.allphotonic import APESimulation

fixed_params = {
    "network": {"photon_loss": 0.2},
    "emitter":{"emitter_T2":3000, "memory_T2":20e+06,"QFC_loss":0},    
    "rgs":{"num_branches_half":6,"b0":6,"b1":3}
}
varying_params = {
    "num_repeaters": [1,2,3,4],
    "distance": [10]
}

sim = APESimulation(fixed_params=fixed_params,
                    varying_params=varying_params,
                    iterations=10,
                    min_successful=0,
                    parameter_file="parameters/scenario1.yml",
                    output_dir="results")
data=sim.start()
sim.plot(final_data=data, show_theo_rate=True, x_axis1="num_repeaters", y_axis1="sim_rate", xlabel1="Number of repeaters", ylabel1="Rate (Hz)",
     label_param1="distance", title1="Rate vs Number of repeaters",
     x_axis2="num_repeaters", y_axis2="fidelity", xlabel2="Number of repeaters", ylabel2="Fidelity",
     label_param2="distance", title2="Fidelity vs Number of repeaters"
     )
sim.finalize()

You can see the entanglement fidelity is affected by the emitter and memory decoherence. In general, the fidelity will decrease with the number of repeaters since each additional repeater will introduce an extra graph state genearation process. The RGS has certain error correcting ability by taking majority votes among the measurement outcomes of all the subtrees. A larger RGS has a better error correction ability but also requires longer genration time and hence receives more decohenrence noise from the emitter. In overall, the optimized RGS size would be a complicated function of the total distance, number of repeaters, RGS generation circuit, all noise parameters and loss parameters.  

# Exercise: Comparing different RGS sizes
Under the quantum repeater network configuration with total distance 10km, fiber attenuation rate=0.2dB/km, emitter coherence time=$3\mu s$ and memory decoherece time=$20ms$, find the optimal RGS size among the given parameters:

| m | $b_0$ | $b_1$ |
|---|---|---|
| 2 | 18 | 1 |
| 3 | 6 | 3 |
| 4 | 4 | 3 |

Task:
For each of the rgs parameters, compute the rate and fidelity and see which gives the best value.

In [ ]:
from qnpack.APE.allphotonic import APESimulation

fixed_params = {
    "network": {"photon_loss": 0.2},
    "emitter":{"emitter_T2":3000, "memory_T2":20e+06,"QFC_loss":0},    
    #Fill in the rgs parameters that you want to calculate here
}
varying_params = {
    "num_repeaters": [1,2,3,4],
    "distance": [10]
}

sim = APESimulation(fixed_params=fixed_params,
                    varying_params=varying_params,
                    iterations=10,
                    min_successful=0,
                    parameter_file="parameters/scenario1.yml",
                    output_dir="results")
data=sim.start()
sim.plot(final_data=data, show_theo_rate=True, x_axis1="num_repeaters", y_axis1="sim_rate", xlabel1="Number of repeaters", ylabel1="Rate (Hz)",
     label_param1="distance", title1="Rate vs Number of repeaters",
     x_axis2="num_repeaters", y_axis2="fidelity", xlabel2="Number of repeaters", ylabel2="Fidelity",
     label_param2="distance", title2="Fidelity vs Number of repeaters"
     )
sim.finalize()